In [1]:
!pip install transformers datasets accelerate

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

2025-12-20 23:38:58.443185: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766273938.583609      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766273938.621249      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766273938.943212      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766273938.943247      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766273938.943250      24 computation_placer.cc:177] computation placer alr

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# bf16 works on T4
use_bf16 = torch.cuda.is_available()

Using device: cuda


In [4]:
dataset = load_dataset("flytech/python-codes-25k")["train"]

# Subsample for Colab
dataset = dataset.shuffle(seed=42).select(range(5000))

# Train / test split
split = dataset.train_test_split(test_size=0.1)

train_dataset = split["train"]
eval_dataset = split["test"]

README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

In [5]:
import random

print(dataset)
for i in range(3):
    k = random.randint(1,1000)
    print("---- Sample", k, "----")
    print(train_dataset[k]["text"])
    print("\n\n")

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 5000
})
---- Sample 259 ----
Download Slack Downloading Slack... ```python
import urllib.request
urllib.request.urlretrieve('https://downloads.slack-edge.com/releases_x64/SlackSetup.exe', 'SlackSetup.exe')
```



---- Sample 551 ----
Create a SQLite database and populate it with some data. Creating a SQLite database and populating it with data... ```python
import sqlite3

conn = sqlite3.connect('mydatabase.db')
cursor = conn.cursor()
cursor.execute('CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, salary REAL)')
cursor.execute('INSERT INTO employees VALUES (1, 'Alice', 50000)')
# Let me run it for you!

conn.commit()
conn.close()
```



---- Sample 511 ----
Question 1: What is the best beginner's book on Python?
Question 2: What is the good book to learn python? I am beginner
 (1). no. (2). yes.
Are questions 1 and 2 asking the same thing? Prepping the toolkit, every tool has its place! Hello li

In [6]:
print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

Train size: 4500
Eval size: 500


In [7]:
print("=== TRAIN SAMPLE ===")
print(train_dataset[0]["text"])

print("=== EVAL SAMPLE ===")
print(eval_dataset[0]["text"])

=== TRAIN SAMPLE ===
Generate a random password using Python I'm on top of it! No need to worry! ```python
import random

password_characters = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%^&*().,?0123456789'

def generate_password(length):
    return ''.join(random.choices(password_characters, k=length))
  
print(generate_password(10))
```
=== EVAL SAMPLE ===
Open Quora for a quick Q&A session in the morning! It's morning, time for a quick Q&A... ```python
from datetime import datetime
import webbrowser
if datetime.now().hour >= 6 and datetime.now().hour <= 9:
    webbrowser.open('https://www.quora.com/')
```


In [8]:
model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Required for causal LM padding
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # IMPORTANT: labels must be provided for causal LM loss
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(tokenized_train)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4500
})


In [10]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if use_bf16 else torch.float32
)

model.gradient_checkpointing_enable()
model.to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

If you want to use `XLMRobertaLMHeadModel` as a standalone, add `is_decoder=True.`


XLMRobertaForCausalLM(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True

In [11]:
training_args = TrainingArguments(
    output_dir="./fft_roberta",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    learning_rate=5e-5,
    optim="adamw_torch",
    bf16=use_bf16,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none",   # can be "wandb" later
    remove_unused_columns=False,

)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer
    
)

/tmp/ipykernel_24/3409326347.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
def generate_code(prompt, model, tokenizer, max_new_tokens=120):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id
        )

    # 🔑 IMPORTANT: decode ONLY the newly generated tokens
    generated_tokens = output_ids[0][input_len:]
    
    #decoded = tokenizer.decode(tokenizer[0]["input_ids"])
    #print(decoded)
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

“Since the baseline model is not instruction-tuned, evaluation prompts were provided in code-continuation format rather than natural-language instructions. This ensures that the generation task aligns with the model’s pretraining and fine-tuning objective.”

In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,3.288700
100,1.041400
150,0.585600
200,0.399400
250,0.268800
300,0.231500
350,0.170300
400,0.139100
450,0.130600
500,0.115000


TrainOutput(global_step=2250, training_loss=0.18092968898349338, metrics={'train_runtime': 965.5398, 'train_samples_per_second': 9.321, 'train_steps_per_second': 2.33, 'total_flos': 1187455749120000.0, 'train_loss': 0.18092968898349338, 'epoch': 2.0})

In [15]:
prompt = "Create a python program to calculate Fibonacci sequence."
prompt2 = """
def is_prime(n):
    \"\"\"Check whether a number is prime\"\"\"
"""

print("=== FINE-TUNED MODEL OUTPUT (prompt) ===")
print(generate_code(prompt, model, tokenizer))

print("=== FINE-TUNED MODEL OUTPUT (prompt2) ===")
print(generate_code(prompt2, model, tokenizer))

=== FINE-TUNED MODEL OUTPUT (prompt) ===
...
=== FINE-TUNED MODEL OUTPUT (prompt2) ===
",, 2016, 2016 2016) 2015) 2017) 2017 2017. 2018))


In [16]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.011926637962460518, 'eval_runtime': 16.1224, 'eval_samples_per_second': 31.013, 'eval_steps_per_second': 3.908, 'epoch': 2.0}
